In [56]:
import torch
import numpy as np
import numpy.random as rn
import scipy.stats as st
from sklearn.metrics import precision_score, recall_score, f1_score

from BPTF import BPTF
from utils_bptf import parafac
from utils import info_rate, init_missing_data

In [2]:
def generate(shp=(30, 30, 20, 10), K=5, alpha=0.1, beta=0.1):
    """Generate a count tensor from the BPTF model.

    PARAMS:
    shp -- (tuple) shape of the generated count tensor
    K -- (int) number of latent components
    alpha -- (float) shape parameter of gamma prior over factors
    beta -- (float) rate parameter of gamma prior over factors

    RETURNS:
    Mu -- (np.ndarray) true Poisson rates
    Y -- (np.ndarray) generated count tensor
    """
    Theta_DK_M = [rn.gamma(alpha, 1./beta, size=(D, K)) for D in shp]
    Mu = parafac(Theta_DK_M)
    assert Mu.shape == shp
    Y = rn.poisson(Mu)
    return Mu, Y

In [4]:
Mu, Y = generate(K=5, alpha=0.1)

In [8]:
Mu.shape

(30, 30, 20, 10)

In [54]:
def load_data(data_dir,seed = None):
    if seed != None:
        np.random.seed(seed)
    held_out_forecast_steps = 2
    held_out_smooth_percent = 0.1
    org_data = np.load(data_dir)
    org_data = org_data['data']
    data_forecast = org_data[-held_out_forecast_steps:, :]
    data_smooth = org_data[:-held_out_forecast_steps, :]

    (smooth_time, dim) = data_smooth.shape
    mask = (np.random.random(size = (smooth_time, dim)) < held_out_smooth_percent).astype(bool)
    masked_train_data = np.ma.array(data_smooth, mask = mask)
    init_data = np.ascontiguousarray(init_missing_data(masked_train_data))
    return init_data, data_forecast, data_smooth, mask # Note that the shapes of init_data and mask are same with data_smooth

In [57]:
import sys
import os
from path import Path
current_directory = os.getcwd()
root_directory = Path(current_directory).parent.parent.parent
sys.path.append(root_directory)

data_dir = root_directory + '/data/icews_preprocessed.npz'
init_data, data_forecast, data_smooth, mask = load_data(data_dir, seed = 1234)

In [60]:
data_forecast = data_forecast.flatten()
fs_nbrgds_forecast_ir = info_rate(data_forecast, data_forecast+ )
print('Information rate of PRGDS for forecast data:', fs_nbrgds_forecast_ir)

Information rate of PRGDS for forecast data: 9.89515660755745


In [34]:
bptf = BPTF(n_modes=init_data.ndim,
            n_components=100,
            max_iter = 800,
            tol = 1e-4,
            smoothness = 100,
            verbose = False,
            alpha = 0.1,
            debug = False)
bptf.fit(init_data)
rc_data = bptf.reconstruct() # estimation value

In [35]:
smooth_truth_count = torch.tensor(data_smooth[mask].astype(np.float32))
bptf_smooth_ir = info_rate(smooth_truth_count, [rc_data[mask]])

In [36]:
bptf_smooth_ir

0.4153704745998122

##########################################